# Airborne Magnetics Inversion — Equivalent Source + Full 3D

This notebook demonstrates the two-stage magnetics inversion workflow:

1. **Equivalent source inversion** — fits TMI flight-line data with a thin layer of dipoles, 
   then predicts Bx/By/Bz/TMI on a uniform grid.
2. **Full 3D inversion** — recovers a 3D susceptibility model from the gridded fields.

Everything runs on **synthetic data** (no real survey required), generated by forward-
modelling a known prismatic susceptibility body.

## Setup

Install the library (from the repo root):

```bash
pip install -e deps/mag_inversion/
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

from SimPEG.potential_fields import magnetics
from SimPEG import maps
from discretize import TensorMesh, TreeMesh
from discretize.utils import mesh_builder_xyz, refine_tree_xyz, active_from_xyz

from mag_inversion import MagEquivalentSourceSystem, MagInversion3DSystem

## 1  Create synthetic survey and true model

We place a rectangular prism with susceptibility χ = 0.1 SI at depth 100–200 m,
spanning a 400 × 400 m block.  Flight lines run E–W at 50 m spacing, 60 m AGL.

In [ ]:
# ── Survey geometry ───────────────────────────────────────────────────────────
FLIGHT_ALT  = 60.0     # metres above ground
LINE_SPACE  = 100.0    # line spacing (m)
SOUNDING_DX = 20.0     # along-line sampling (m)
SURVEY_HALF = 1000.0   # half-width of survey area (m)

xi_lines = np.arange(-SURVEY_HALF, SURVEY_HALF + LINE_SPACE, LINE_SPACE)
x_snd    = np.arange(-SURVEY_HALF, SURVEY_HALF + SOUNDING_DX, SOUNDING_DX)

X_all, Y_all = [], []
line_ids = []
for li, y_line in enumerate(xi_lines):
    X_all.append(x_snd)
    Y_all.append(np.full_like(x_snd, y_line))
    line_ids.extend([li] * len(x_snd))

X_all = np.concatenate(X_all)
Y_all = np.concatenate(Y_all)
Z_all = np.full_like(X_all, FLIGHT_ALT)
receiver_locs = np.c_[X_all, Y_all, Z_all]

print(f"Survey: {len(receiver_locs)} soundings over {len(xi_lines)} lines")

In [ ]:
# ── Earth field ───────────────────────────────────────────────────────────────
FIELD_INTENSITY  = 52000.0   # nT
FIELD_INCL       = 65.0      # degrees
FIELD_DECL       = 5.0       # degrees
FIELD_PARAMS     = [FIELD_INTENSITY, FIELD_INCL, FIELD_DECL]

# ── True model mesh (dense TensorMesh for forward modelling) ─────────────────
cs_true = 25.0
hx = np.ones(int(2 * SURVEY_HALF / cs_true) + 20) * cs_true
hy = hx.copy()
hz = np.ones(20) * cs_true           # 0 – 500 m depth

mesh_true = TensorMesh(
    [hx, hy, hz],
    origin=[-SURVEY_HALF - 10*cs_true, -SURVEY_HALF - 10*cs_true, -500]
)

# ── Susceptibility prism ──────────────────────────────────────────────────────
chi_true = np.zeros(mesh_true.nC)
cc = mesh_true.cell_centers
prism_mask = (
    (np.abs(cc[:, 0]) < 200) &
    (np.abs(cc[:, 1]) < 200) &
    (cc[:, 2] > -200) &
    (cc[:, 2] < -100)
)
chi_true[prism_mask] = 0.1

print(f"True mesh: {mesh_true.nC} cells,  prism cells: {prism_mask.sum()}")

In [ ]:
# ── Forward model synthetic TMI ───────────────────────────────────────────────
receiver_fwd  = magnetics.receivers.Point(receiver_locs, components=["tmi"])
source_fwd    = magnetics.sources.SourceField(receiver_list=[receiver_fwd], parameters=FIELD_PARAMS)
survey_fwd    = magnetics.survey.Survey(source_fwd)

sim_fwd = magnetics.simulation.Simulation3DIntegral(
    mesh=mesh_true,
    survey=survey_fwd,
    chiMap=maps.IdentityMap(nP=mesh_true.nC),
    store_sensitivities="forward_only",
)

tmi_clean = sim_fwd.dpred(chi_true)

# Add Gaussian noise (1% + 1 nT floor)
np.random.seed(42)
noise   = 0.01 * np.abs(tmi_clean) + 1.0
tmi_obs = tmi_clean + noise * np.random.randn(len(tmi_clean))

print(f"Peak TMI anomaly: {tmi_clean.max():.1f} nT")

plt.figure(figsize=(8, 4))
sc = plt.scatter(X_all, Y_all, c=tmi_obs, s=1, cmap='RdBu_r')
plt.colorbar(sc, label='TMI (nT)')
plt.title('Synthetic observed TMI')
plt.xlabel('Easting (m)'); plt.ylabel('Northing (m)')
plt.axis('equal'); plt.tight_layout(); plt.show()

## 2  Build a mock MagData object

MagEquivalentSourceSystem expects an AirMagTools.MagData instance.  For this
standalone example we create a minimal duck-typed object.

In [ ]:
import pandas as pd

class MockMagData:
    """Minimal MagData-compatible object for standalone testing."""
    def __init__(self, x, y, alt, tmi, line_ids, meta):
        index = pd.MultiIndex.from_arrays([line_ids, np.arange(len(x))], names=['line', 'idx'])
        self.data = pd.DataFrame({'UTMX': x, 'UTMY': y, 'alt': alt, 'tmi': tmi}, index=index)
        self.meta = meta

mag_data = MockMagData(
    x=X_all, y=Y_all, alt=Z_all, tmi=tmi_obs,
    line_ids=line_ids,
    meta={
        'crs': 'EPSG:32632',
        'field_intensity':   FIELD_INTENSITY,
        'field_inclination': FIELD_INCL,
        'field_declination': FIELD_DECL,
    }
)

## 3  Equivalent Source Inversion

In [ ]:
equiv = MagEquivalentSourceSystem(
    mag_data,
    layer__cell_size=50,           # 50 m horizontal cell size
    layer__depth_below_flight=30,  # layer 30 m below min flight altitude
    store_sensitivities="ram",     # small survey: keep G in RAM
    optimizer__max_iter=20,
    output__components=["tmi", "bx", "by", "bz"],
)

In [ ]:
# Run the inversion
model_equiv, mesh_equiv, sim_equiv = equiv.invert()

In [ ]:
# Convert to xarray Dataset (gridded fields at mean flight altitude)
ds_equiv = equiv.to_xarray(model_equiv, mesh_equiv, sim_equiv)
print(ds_equiv)

# Visualise
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, comp in zip(axes, ['tmi', 'bx', 'by', 'bz']):
    if comp in ds_equiv:
        ds_equiv[comp].plot(ax=ax, cmap='RdBu_r')
        ax.set_title(comp.upper())
        ax.set_aspect('equal')
plt.tight_layout()
plt.suptitle('Equivalent source — gridded output', y=1.02)
plt.show()

In [ ]:
# Optional: save to webxtile for inspection / downstream use
# ds_equiv.webxtile.to_webxtile("/tmp/equiv_source_output/")

## 4  Full 3D Inversion

We pass the equivalent source output directly to MagInversion3DSystem.
For amplitude inversion (model_type='amplitude') the system computes |B|
from Bx/By/Bz and inverts for scalar susceptibility in a way that is
insensitive to remanent magnetisation direction.

In [ ]:
inv3d = MagInversion3DSystem(
    ds_equiv,
    model_type="scalar",           # "scalar", "vector", or "amplitude"
    mesh__core_cell_size=50,
    mesh__depth_core=500,
    store_sensitivities="ram",     # small survey: keep in RAM
    optimizer__max_iter=15,
    directives__sensitivity_weights__enable=True,
)

In [ ]:
model_3d, mesh_3d, sim_3d, active_3d = inv3d.invert()

In [ ]:
ds_3d = inv3d.to_xarray(model_3d, mesh_3d, active_3d)
print(ds_3d)

In [ ]:
# Plot horizontal slice at depth ~150 m (centre of true prism)
chi = ds_3d['susceptibility']

# Find z slice closest to -150 m
z_target = -150.0
z_idx = int(np.argmin(np.abs(chi.z.values - z_target)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

chi.isel(z=z_idx).plot(ax=axes[0], cmap='viridis')
axes[0].set_title(f'Recovered susceptibility  z ≈ {chi.z.values[z_idx]:.0f} m')
axes[0].set_aspect('equal')

# E–W vertical section through y=0
y_idx = int(np.argmin(np.abs(chi.y.values - 0)))
chi.isel(y=y_idx).plot(ax=axes[1], cmap='viridis', yincrease=False)
axes[1].set_title(f'Vertical section  y ≈ {chi.y.values[y_idx]:.0f} m')

plt.tight_layout()
plt.show()

In [ ]:
# Optional: save 3D result to webxtile
# ds_3d.webxtile.to_webxtile("/tmp/mag3d_output/")

## 5  Customisation Reference

All parameters are class-level attributes that can be overridden at instantiation.
Here is a summary of the most commonly changed ones:

### MagEquivalentSourceSystem

| Attribute | Default | Description |
|-----------|---------|-------------|
| `columns__x` | `"UTMX"` | Easting column name |
| `columns__y` | `"UTMY"` | Northing column name |
| `columns__altitude` | `"alt"` | Flight altitude column |
| `columns__tmi` | `"tmi"` | TMI data column |
| `columns__tmi_std` | `None` | Uncertainty column (or derived) |
| `field_intensity` | from meta | Earth field nT |
| `field_inclination` | from meta | Earth field inclination deg |
| `field_declination` | from meta | Earth field declination deg |
| `layer__depth_below_flight` | `30.0` | Layer depth (m) |
| `layer__cell_size` | `50.0` | Horizontal cell size (m) |
| `layer__cell_thickness` | `5.0` | Layer thickness (m) |
| `layer__padding_cells` | `8` | Padding cells each side |
| `model_type` | `"scalar"` | `"scalar"` or `"vector"` |
| `uncertainties__std_data` | `0.02` | Fractional uncertainty |
| `uncertainties__floor_nT` | `1.0` | Noise floor (nT) |
| `regularization__alpha_s` | `1e-4` | Smallness weight |
| `regularization__alpha_x` | `1.0` | X smoothness |
| `regularization__alpha_y` | `1.0` | Y smoothness |
| `optimizer__max_iter` | `40` | Max GN iterations |
| `directives__irls__enable` | `False` | Enable sparse inversion |
| `store_sensitivities` | `"disk"` | G storage mode |
| `sensitivity_path` | tmp dir | Root path for G cache |
| `output__altitude` | mean alt | Prediction altitude (m) |
| `output__xy_spacing` | cell size | Output grid spacing (m) |
| `output__components` | all | Components to predict |

### MagInversion3DSystem

| Attribute | Default | Description |
|-----------|---------|-------------|
| `model_type` | `"scalar"` | `"scalar"`, `"vector"`, or `"amplitude"` |
| `mesh__core_cell_size` | `50.0` | OcTree core cell size (m) |
| `mesh__octree_levels_obs` | `[2, 4]` | Refinement near observations |
| `mesh__octree_levels_surf` | `[2, 4]` | Refinement near surface |
| `mesh__depth_core` | `500.0` | Core depth (m) |
| `mesh__max_distance` | `5000.0` | Horizontal padding (m) |
| `topography` | None | Nx3 topo array or DataArray |
| `startmodel__susceptibility` | `1e-4` | Starting susceptibility |
| `regularization__alpha_s` | `1e-4` | Smallness weight |
| `regularization__alpha_[xyz]` | `1.0` | Smoothness weights |
| `directives__sensitivity_weights__enable` | `True` | Depth weighting |
| `directives__irls__enable` | `False` | Sparse inversion |
| `store_sensitivities` | `"disk"` | G storage mode |
| `sensitivity_path` | tmp dir | Root path for G cache |
| `output__xy_spacing` | core cell | Output grid XY spacing |
| `output__z_spacing` | core cell | Output grid Z spacing |